# Data cleaning

Reads the raw main-tour and qualifying/challenger match CSVs, aligns their
schemas, coerces types, imputes missing values, and writes
`atp_matches_data_cleaned.csv` for `feature_add.ipynb` to consume next.

## 1. Load raw data

In [1]:
# Raw source data: main-tour matches and qualifying/challenger matches ship as
# separate files with slightly different schemas; both get aligned and merged below.
import pandas as pd
import numpy as np

df = pd.read_csv("atp_matches_git.csv")
df_qual = pd.read_csv("atp_qual.csv")

C:\Users\rohan\AppData\Local\Temp\ipykernel_29476\1061201531.py:6: DtypeWarning: Columns (7,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("atp_matches_git.csv")
C:\Users\rohan\AppData\Local\Temp\ipykernel_29476\1061201531.py:7: DtypeWarning: Columns (50) have mixed types. Specify dtype option on import or set low_memory=False.
  df_qual = pd.read_csv("atp_qual.csv")


## 2. Basic hygiene

Both CSVs occasionally repeat their header row as a data row (e.g. from
concatenating yearly files); drop those, and strip stray whitespace from
every text column.

In [2]:
df = df[df["tourney_id"].ne("tourney_id")].copy()

# Strip whitespace in all object columns
obj_cols = df.select_dtypes(include="object").columns
df[obj_cols] = df[obj_cols].apply(lambda s: s.str.strip())

df_qual = df_qual[df_qual["tourney_id"].ne("tourney_id")].copy()

# Strip whitespace in all object columns
obj_cols = df_qual.select_dtypes(include="object").columns
df_qual[obj_cols] = df_qual[obj_cols].apply(lambda s: s.str.strip())

## 3. Type coercion

### 3a. Main-tour file

CSV columns come in as strings; coerce the numeric ones and normalize `surface`
(Carpet is rare enough in this era of tennis to fold into Hard).

In [3]:
# Columns that should be numeric (loaded as strings/objects by default,
# since the raw CSVs mix header re-reads and blank values into these columns).
numeric_cols = [
    "draw_size", "match_num", "best_of", "minutes",
    "winner_id", "winner_seed", "winner_ht", "winner_age",
    "loser_id", "loser_seed", "loser_ht", "loser_age",
    "winner_rank", "winner_rank_points", "loser_rank", "loser_rank_points",
    # post-match stats (keep numeric even if you later drop for leakage)
    "w_ace","w_df","w_svpt","w_1stIn","w_1stWon","w_2ndWon","w_SvGms","w_bpSaved","w_bpFaced",
    "l_ace","l_df","l_svpt","l_1stIn","l_1stWon","l_2ndWon","l_SvGms","l_bpSaved","l_bpFaced", "match_date",
]

for c in numeric_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

df["surface"] = df["surface"].replace({"Carpet": "Hard"})


### 3b. Qualifying/challenger file

Same treatment as the main-tour file, plus aligning its date column to match.

In [4]:
df_qual = df_qual.drop('match_date', axis=1)
df_qual = df_qual.rename(columns={'match_date_yyyymmdd': 'match_date'})
for c in numeric_cols:
    if c in df_qual.columns:
        df_qual[c] = pd.to_numeric(df_qual[c], errors="coerce")

df_qual["surface"] = df_qual["surface"].replace({"Carpet": "Hard"})

# A handful of qualifying rows have no recorded score. Fill with a placeholder
# (rather than leaving NaN) so later string checks like `.str.contains("RET")`
# don't need to special-case nulls; the placeholder never matches those patterns.
df_qual["score"] = df_qual["score"].fillna("ddd")

## 4. Merge

Combine both files into a single chronologically-ordered match history.

In [5]:
# Combine main-tour and qualifying/challenger matches into one match history.
# match_date is imputed and the frame is sorted into chronological order in the
# next two cells, after both files are merged (so the imputation has the widest
# possible sample to learn each tournament level's round-offset pattern from).
df = pd.concat([df, df_qual], ignore_index=True)

In [6]:
# match_date is missing for ~35% of rows -- mostly Challenger-level matches,
# which have zero populated match_date anywhere in this dataset. Estimate the
# missing ones from tourney_date + round, using the median
# (match_date - tourney_date) day-offset observed for each (tourney_level, round)
# group among the rows where match_date IS known (e.g. a Grand Slam Final is
# ~13 days after tourney_date; a tour-level R32 is ~1 day after). Falls back to
# the A-level curve for Challenger (no ground truth of its own, but structurally
# the same 32-draw format), then the round's global median across all levels,
# then 0 (i.e. just tourney_date) if a round is never seen at all.
_tourney_dt = pd.to_datetime(df["tourney_date"], format="%Y%m%d")
_match_dt = pd.to_datetime(df["match_date"], format="%Y%m%d")
_offset_days = (_match_dt - _tourney_dt).dt.days

_group_medians = _offset_days.groupby([df["tourney_level"], df["round"]]).median()
_group_counts = _offset_days.groupby([df["tourney_level"], df["round"]]).count()
_global_round_medians = _offset_days.groupby(df["round"]).median()

MIN_SAMPLES = 5

def _offset_for(level, round_):
    key = (level, round_)
    if key in _group_medians.index and _group_counts.loc[key] >= MIN_SAMPLES:
        return _group_medians.loc[key]
    if level == "C" and ("A", round_) in _group_medians.index:
        return _group_medians.loc[("A", round_)]
    if round_ in _global_round_medians.index:
        return _global_round_medians.loc[round_]
    return 0.0

# Track which rows were actually estimated (for transparency/debugging),
# before match_date itself gets overwritten below.
df["match_date_is_estimated"] = df["match_date"].isna()

_missing = df["match_date"].isna()
_offsets = df.loc[_missing].apply(lambda r: _offset_for(r["tourney_level"], r["round"]), axis=1)
_imputed_dt = _tourney_dt.loc[_missing] + pd.to_timedelta(_offsets, unit="D")

# Keep the on-disk format identical to the existing YYYYMMDD-numeric convention,
# so downstream code (main.py's format='%Y%m%d' parse) needs no changes.
df.loc[_missing, "match_date"] = _imputed_dt.dt.strftime("%Y%m%d").astype(float)

print(f"Imputed match_date for {_missing.sum()} of {len(df)} rows ({_missing.mean():.1%}).")

Imputed match_date for 18823 of 53367 rows (35.3%).


In [7]:
# Stable ("mergesort") sort on (tourney_date, match_num). match_num is a real
# observed ordinal field (populated for 99%+ of rows) that correctly orders
# matches within a tournament (e.g. R32 before QF before F). Sorting on
# tourney_date alone previously left same-day matches in raw file order, which
# turned out to be *descending* match_num (Final listed before Semifinal...) --
# since feature_add.ipynb processes rows in this order to compute Elo, that bug
# meant a player's pre-match Elo for an earlier round could already reflect a
# later round's result within the same tournament. match_num is used as the
# tiebreaker here rather than the match_date computed above, since a flat
# per-round offset estimate ties across every match in the same round, whereas
# match_num never does.
df = df.sort_values(["tourney_date", "match_num"], kind="mergesort").reset_index(drop=True)

## 5. Impute missing values

Player bio fields (height/age/hand) are filled per-player using known values from
their other matches; match stats are filled per `best_of` group; anything still
missing falls back to the global median/mode.

In [8]:
# -------- Player bio cleanup --------
# Treat impossible heights as missing before imputation.
HEIGHT_COLS = ["winner_ht", "loser_ht"]
HEIGHT_MIN_CM, HEIGHT_MAX_CM = 140, 220
for col in HEIGHT_COLS:
    df.loc[~df[col].between(HEIGHT_MIN_CM, HEIGHT_MAX_CM), col] = np.nan

# Keep ffill + bfill inside each player group using transform(...).
def fill_player_numeric(frame, player_col, value_col):
    frame[value_col] = frame.groupby(player_col)[value_col].transform(lambda s: s.ffill().bfill())
    frame[value_col] = frame[value_col].fillna(frame[value_col].median())

for value_col in ["winner_age", "winner_ht"]:
    fill_player_numeric(df, "winner_name", value_col)

for value_col in ["loser_age", "loser_ht"]:
    fill_player_numeric(df, "loser_name", value_col)

# Fill missing handedness with mode; fallback to right-handed if empty.
for col in ["winner_hand", "loser_hand"]:
    mode = df[col].mode(dropna=True)
    fill_val = mode.iloc[0] if not mode.empty else "R"
    df[col] = df[col].fillna(fill_val)

# -------- Match stat cleanup --------
# Fill post-match stats within best_of groups, then fallback to global median.
POST_MATCH_STATS = [
    "ace", "df", "svpt", "1stIn", "1stWon",
    "2ndWon", "bpSaved", "bpFaced", "SvGms"
]
for stat in POST_MATCH_STATS:
    for prefix in ["w_", "l_"]:
        col = f"{prefix}{stat}"
        if col in df.columns:
            df[col] = df.groupby("best_of")[col].transform(lambda s: s.fillna(s.median()))
            df[col] = df[col].fillna(df[col].median())

# Match duration and categorical fallback.
df["minutes"] = df.groupby("best_of")["minutes"].transform(lambda s: s.fillna(s.median()))
df["surface"] = df["surface"].fillna("Hard")

# Rank points: use nearest known value within each player history.
for player_col, rank_col in [
    ("winner_name", "winner_rank_points"),
    ("loser_name", "loser_rank_points"),
]:
    df[rank_col] = df.groupby(player_col)[rank_col].transform(lambda s: s.ffill().bfill())


## 6. Missing-value audit

Confirm the imputation above actually closed the gaps before relying on it.

In [9]:
# No missing values left
missing_summary = (
    df.isna()
      .mean()
      .sort_values(ascending=False)
      .to_frame("missing_frac")
)

missing_summary

,missing_frac
winner_entry,0.841344
loser_entry,0.739521
loser_seed,0.715723
winner_seed,0.529784
winner_id,0.362171
loser_id,0.362115
match_day_offset,0.352709
loser_rank,0.019282
loser_rank_points,0.011093
match_num,0.009163


## 7. Validate

Sanity-check the cleaned frame before trusting it downstream.

In [10]:
def test_cleaned_df(df):
    """Sanity-check the cleaned dataframe before export."""
    required_cols = [
        "tourney_id", "tourney_date", "surface", "winner_name", "loser_name",
        "best_of", "score", "winner_age", "loser_age", "winner_ht", "loser_ht",
        "match_date",
    ]
    missing_cols = [c for c in required_cols if c not in df.columns]
    assert not missing_cols, f"Missing expected columns: {missing_cols}"

    # Key columns should have no nulls after cleaning
    for col in required_cols:
        n_missing = df[col].isna().sum()
        assert n_missing == 0, f"{col} has {n_missing} missing values"

    assert set(df["surface"].unique()) <= {"Hard", "Clay", "Grass"}, "Unexpected surface value"
    assert (df["best_of"].isin([3, 5])).all(), "best_of contains values other than 3 or 5"
    assert len(df) > 0, "Cleaned dataframe is empty"

    print(f"All checks passed on {len(df)} rows.")


test_cleaned_df(df)


All checks passed on 53367 rows.


## 8. Export

Only runs if the checks above pass.

In [11]:
# Feeds directly into feature_add.ipynb — keep this filename in sync with it.
df.to_csv('atp_matches_data_cleaned.csv', index=False, header=True)